In [89]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import make_column_transformer
import pandas as pd

url = "https://learn.zone01oujda.ma/api/content/root/01-edu_module/content/pipeline/data/breast-cancer.csv"

cols = [
    "age", "menopause", "tumor-size", "inv-nodes",
    "node-caps", "deg-malig", "breast", "breast-quad",
    "irradiat", "Class"
]

df = pd.read_csv(url, header=None, names=cols)
df.drop(columns="Class", inplace=True)

df.dropna(inplace=True)

X_train, X_test = train_test_split(
    df,
    test_size=0.2,
    random_state=43
)

print(X_train.nunique())

df.head()

age             6
menopause       3
tumor-size     11
inv-nodes       6
node-caps       2
deg-malig       3
breast          2
breast-quad     5
irradiat        2
dtype: int64


,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat
0,40-49,premeno,15-19,0-2,yes,3,right,left_up,no
1,50-59,ge40,15-19,0-2,no,1,right,central,no
2,50-59,ge40,35-39,0-2,no,2,left,left_low,no
3,40-49,premeno,35-39,0-2,yes,3,right,left_low,yes
4,40-49,premeno,30-34,3-5,yes,2,left,right_up,no


In [90]:
ohe_cols = ['node-caps', 'breast', 'breast-quad', 'irradiat']

ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

ohe.fit(X_train[ohe_cols])
print(ohe.transform(X_test[ohe_cols])[:10])


[[1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0.]
 [1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]]


In [91]:
ord_cols = ["menopause", "age", "tumor-size", "inv-nodes", "deg-malig"]

ordinal_categories = [
    ['lt40', 'premeno', 'ge40'],
    ['10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99'],
    ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59'],
    ['0-2','3-5','6-8','9-11','12-14','15-17','18-20','21-23','24-26','27-29','30-32','33-35','36-39'],
    [1, 2, 3]
]

oe = OrdinalEncoder(categories=ordinal_categories)

oe.fit(X_train[ord_cols])
oe.transform(X_test[ord_cols])[:10]

array([[2., 5., 2., 0., 1.],
       [2., 5., 2., 0., 0.],
       [2., 5., 4., 5., 2.],
       [1., 4., 5., 1., 1.],
       [2., 5., 5., 0., 2.],
       [1., 2., 1., 0., 1.],
       [1., 2., 8., 0., 1.],
       [2., 5., 2., 0., 0.],
       [2., 5., 5., 0., 2.],
       [1., 2., 3., 0., 0.]])

In [92]:
preprocessor = make_column_transformer(
    (ohe, ohe_cols),
    (oe, ord_cols),
    remainder="drop"
)

preprocessor.fit(X_train)
X_test_final = preprocessor.transform(X_test)
X_test_final[:2]

array([[1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 2., 5., 2., 0., 1.],
       [1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 2., 5., 2., 0., 0.]])